## Data Tokenizer

### Import data "the-verdict" from kaggle

In [2]:
import kagglehub
path = kagglehub.dataset_download("golammostofas/the-verdict", output_dir="data/the-verdict/")
print(path)

data/the-verdict/


### Process the raw data file 

In [3]:
with open("data/the-verdict/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total characters: ", len(raw_text))
print(raw_text[:50])

Total characters:  20479
I HAD always thought Jack Gisburn rather a cheap g


### Tokenize words/sub-words in the book

In [10]:
import re

sample_text = """
Hello World! this is awesome. [link](http://link.com)
This is line 2, and I am interested to know what happens when line ends.
"""

# result = re.split(r'(\s)', sample_text) #splits by white space
result = [t for t in re.split(r'([,.!]|\s)', sample_text.strip()) if t]
print(result)

['Hello', ' ', 'World', '!', ' ', 'this', ' ', 'is', ' ', 'awesome', '.', ' ', '[link](http://link', '.', 'com)', '\n', 'This', ' ', 'is', ' ', 'line', ' ', '2', ',', ' ', 'and', ' ', 'I', ' ', 'am', ' ', 'interested', ' ', 'to', ' ', 'know', ' ', 'what', ' ', 'happens', ' ', 'when', ' ', 'line', ' ', 'ends', '.']


In [ ]:
import regex as re

# Suggested by CLAUDE
GPT2_PAT = re.compile(
    r"'(?:[sdmt]|ll|ve|re)"        # contractions: 't, 's, 'll, 've, 're, 'd, 'm
    r"|[^\r\n\p{L}\p{N}]?\p{L}+"   # words, optionally prefixed by a space/symbol
    r"|\p{N}+"                      # numbers
    r"| ?[^\s\p{L}\p{N}]+[\r\n]*"  # punctuation/symbols
    r"|\s+(?!\S)"                   # trailing whitespace
    r"|\s+"                         # remaining whitespace
)

tokens = GPT2_PAT.findall(sample_text)
print(tokens)
# print(len(tokens))


['\n', 'Hello', ' World', '!', ' this', ' is', ' awesome', '.', ' [', 'link', '](', 'http', '://', 'link', '.com', ')\n', 'This', ' is', ' line', ' ', '2', ',', ' and', ' I', ' am', ' interested', ' to', ' know', ' what', ' happens', ' when', ' line', ' ends', '.\n']


In [14]:
# Convert the book to tokens

tokens = GPT2_PAT.findall(raw_text)
print(len(tokens))
print(tokens[:10])


4538
['I', ' HAD', ' always', ' thought', ' Jack', ' Gisburn', ' rather', ' a', ' cheap', ' genius']


In [17]:
all_words = sorted(set(tokens))
vocab = {token:id for id, token in enumerate(all_words)}

In [28]:
print(len(vocab))

counter = 0
for key, value in vocab.items():
    print("{",key, ",", value,"}")
    counter = counter + 1
    if counter > 20:
        break

1269
{  " , 0 }
{  ' , 1 }
{  ( , 2 }
{  . , 3 }
{  .

 , 4 }
{  ."

 , 5 }
{  A , 6 }
{  Among , 7 }
{  And , 8 }
{  Arrt , 9 }
{  At , 10 }
{  Burlington , 11 }
{  But , 12 }
{  By , 13 }
{  Carlo , 14 }
{  Chicago , 15 }
{  Claude , 16 }
{  Croft , 17 }
{  Devonshire , 18 }
{  Don , 19 }
{  Dubarry , 20 }


### Simple Tokenizer class

In [41]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
        self._GPT2_PAT = re.compile(
            r"'(?:[sdmt]|ll|ve|re)"        # contractions: 't, 's, 'll, 've, 're, 'd, 'm
            r"|[^\r\n\p{L}\p{N}]?\p{L}+"   # words, optionally prefixed by a space/symbol
            r"|\p{N}+"                      # numbers
            r"| ?[^\s\p{L}\p{N}]+[\r\n]*"  # punctuation/symbols
            r"|\s+(?!\S)"                   # trailing whitespace
            r"|\s+"                         # remaining whitespace
        )

    def encode(self, text: str) -> list[int]:
        return [self.str_to_int[i] for i in self._GPT2_PAT.findall(text)]
    
    def decode(self, ids: list[int]) -> str:
        return "".join([self.int_to_str[i] for i in ids])
        

In [47]:
tokenizer = SimpleTokenizerV1(vocab)
print(raw_text[:98])
tokens = tokenizer.encode(raw_text[:98])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no


In [45]:
print(tokens)

[1179, 30, 121, 948, 39, 26, 771, 88, 225, 448, 1119, 1263, 88, 462, 398, 356, 1119, 1254, 542, 1019, 663]


In [46]:
decoded_string = tokenizer.decode(tokens)
print(decoded_string)

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no


In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        
        self._unknown_token = " <|unk|>"
        if self._unknown_token not in self.str_to_int:
            self.str_to_int[self._unknown_token] = max(self.str_to_int.values()) + 1
        
        self.int_to_str = {i:s for s,i in self.str_to_int.items()}
        self._GPT2_PAT = re.compile(
            r"'(?:[sdmt]|ll|ve|re)"        # contractions: 't, 's, 'll, 've, 're, 'd, 'm
            r"|[^\r\n\p{L}\p{N}]?\p{L}+"   # words, optionally prefixed by a space/symbol
            r"|\p{N}+"                      # numbers
            r"| ?[^\s\p{L}\p{N}]+[\r\n]*"  # punctuation/symbols
            r"|\s+(?!\S)"                   # trailing whitespace
            r"|\s+"                         # remaining whitespace
        )

    def encode(self, text: str) -> list[int]:
        # print(self._GPT2_PAT.findall(text))
        return [self.str_to_int.get(i, self.str_to_int[self._unknown_token]) 
                for i in self._GPT2_PAT.findall(text)]
    
    def decode(self, ids: list[int]) -> str:
        return "".join([self.int_to_str[i] for i in ids])
        

In [70]:
sample_text_2 = "1000 is greater than 100"
tokenizer2 = SimpleTokenizerV2(vocab)
encoded_string = tokenizer2.encode(sample_text_2)
print(encoded_string)
tokenizer2.decode(encoded_string)

# Real LLMs solve this with BPE (Byte Pair Encoding), 
# which works at the byte level so nothing is ever truly unknown.

['1000', ' is', ' greater', ' than', ' ', '100']
[1270, 541, 1270, 931, 1270, 1270]


' <|unk|> is <|unk|> than <|unk|> <|unk|>'